# Fashion-MNIST y lectura de datos

**Capítulo 1 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_linear-classification/image-classification-dataset.ipynb` · [Lección original](https://d2l.ai/chapter_linear-classification/image-classification-dataset.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# El conjunto de datos de clasificación de imágenes
<a id="sec_fashion_mnist"></a>

(El conjunto de datos MNIST es uno de los conjuntos de datos ampliamente utilizados para la clasificación de imágenes, mientras que es demasiado simple como conjunto de datos de referencia. Usaremos el conjunto de datos Fashion-MNIST similar, pero más complejo)

Un conjunto de datos ampliamente utilizado para la clasificación de imágenes es el [MNIST dataset](https://en.wikipedia.org/wiki/MNIST_database) [LeCun.Bottou.Bengio.ea.1998](https://d2l.ai/chapter_references/zreferences.html) de dígitos escritos a mano. En el momento de su lanzamiento en la década de 1990 planteó un desafío formidable a la mayoría de algoritmos de aprendizaje automático, consistentes en 60.000 imágenes de resolución de píxeles $28 \times 28$ (más un conjunto de datos de prueba de 10.000 imágenes). Para poner las cosas en perspectiva, en 1995, un Sun SPARCStation 5 con una enorme cantidad de 64 MB de RAM y un blisters 5 MPLOPs fue considerado equipo de última generación para el aprendizaje automático en AT&T Bell Laboratories. Lograr una alta precisión en el reconocimiento de dígitos era un componente clave en la automatización de la clasificación de letras para el USPS en la década de 1990. Redes profundas como LeNet-5 [LeCun.Jackel.Bottou.ea.1995](https://d2l.ai/chapter_references/zreferences.html), máquinas vectoriales de soporte con invariantes [Scholkopf.Burges.Vapnik.1996](https://d2l.ai/chapter_references/zreferences.html), y clasificadores de distancia tangentes [Simard.LeCun.Denker.ea.1998](https://d2l.ai/chapter_references/zreferences.html) todos podrían alcanzar tasas de error inferiores al 1%.

Durante más de una década, MNIST sirvió como *el* punto de referencia para comparar algoritmos de aprendizaje automático. Si bien tuvo una buena ejecución como conjunto de datos de referencia, incluso modelos simples para los estándares actuales logran una precisión de clasificación superior al 95%, por lo que no es adecuado para distinguir entre modelos fuertes y más débiles. Aún más, el conjunto de datos permite un *muy* alto nivel de precisión, que normalmente no se ve en muchos problemas de clasificación. Este desarrollo algorítmico sesgado hacia familias específicas de algoritmos que pueden aprovechar conjuntos de datos limpios, como métodos de conjunto activos y algoritmos de conjunto activos que buscan límites. [Deng.Dong.Socher.ea.2009](https://d2l.ai/chapter_references/zreferences.html) Por desgracia, ImageNet es demasiado grande para muchos de los ejemplos e ilustraciones de este libro, ya que tomaría demasiado tiempo entrenar para hacer los ejemplos interactivos. Como un sustituto vamos a centrar nuestra discusión en las próximas secciones en el conjunto de datos cualitativamente similar, pero mucho más pequeño de moda-MNIST [Xiao.Rasul.Vollgraf.2017](https://d2l.ai/chapter_references/zreferences.html) que fue lanzado en 2017. Contiene imágenes de 10 categorías de ropa en $28 \times 28$ resolución de píxeles.


In [ ]:
%matplotlib inline
import time
import torch
import torchvision
from torchvision import transforms
from laboratorio import d2l

d2l.use_svg_display()

## Cargando el conjunto de datos
Dado que el conjunto de datos de Fashion-MNIST es tan útil, todos los marcos principales proporcionan versiones preprocesadas de él. Podemos **descargarlo y leerlo en la memoria utilizando utilidades de framework integradas.**


In [ ]:
class FashionMNIST(d2l.DataModule):  #@save
    """El conjunto de datos de la moda MNIST."""
    def __init__(self, batch_size=64, resize=(28, 28)):
        super().__init__()
        self.save_hyperparameters()
        trans = transforms.Compose([transforms.Resize(resize),
                                    transforms.ToTensor()])
        self.train = torchvision.datasets.FashionMNIST(
            root=self.root, train=True, transform=trans, download=True)
        self.val = torchvision.datasets.FashionMNIST(
            root=self.root, train=False, transform=trans, download=True)
        self.train, self.val = d2l.separar_validacion(self.train)


Fashion-MNIST consiste en imágenes de 10 categorías, cada una representada por 6000 imágenes en el conjunto de datos de entrenamiento y por 1000 en el conjunto de datos de prueba. Para evaluar el rendimiento del modelo se utiliza un *conjunto de datos de prueba* (no debe utilizarse para el entrenamiento), por lo que el conjunto de entrenamiento y el conjunto de pruebas contienen 60.000 y 10.000 imágenes, respectivamente.


In [ ]:
data = FashionMNIST(resize=(32, 32))
len(data.train), len(data.val)

Las imágenes son escala de grises y de alta escala a $32 \times 32$ pixels en la resolución de arriba. Esto es similar al conjunto de datos original del MNIST que consistía en imágenes (binarias) en blanco y negro. Observe, sin embargo, que la mayoría de los datos de imágenes modernas tienen tres canales (rojo, verde, azul) y que las imágenes hiperespectrales pueden tener más de 100 canales (el sensor HyMap tiene 126 canales). $c \times h \times w$ tensor, donde $c$ es el número de canales de color, $h$ es la altura y $w$ es el ancho.


In [ ]:
data.train[0][0].shape

Dos funciones de utilidad para visualizar el conjunto de datos

Las categorías de Fashion-MNIST tienen nombres comprensibles para el ser humano. El siguiente método de conveniencia se convierte entre las etiquetas numéricas y sus nombres.


In [ ]:
@d2l.add_to_class(FashionMNIST)  #@save
def text_labels(self, indices):
    """Devuelve las etiquetas de texto."""
    labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
              'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [labels[int(i)] for i in indices]

## Leyendo un minibatch
Para hacer nuestra vida más fácil al leer de los conjuntos de entrenamiento y pruebas, utilizamos el iterador de datos integrado en lugar de crear uno desde cero. Recordemos que en cada iteración, un iterador de datos **lee un minibatch de datos con tamaño `batch_size`.** También barajamos aleatoriamente los ejemplos para el iterador de datos de entrenamiento.


### Nota docente de Hespérides

Una forma correcta no garantiza un significado correcto: fija qué representa cada eje antes de calcular. En el primer entrenamiento, distingue logits, probabilidades y etiquetas. La ilustración presenta una intuición; el explorador final muestra coordenadas realmente calculadas por una red pequeña.

Vínculo con los apuntes: sesión 1, «Fashion-MNIST y lectura de datos».


In [ ]:
@d2l.add_to_class(FashionMNIST)  #@save
def get_dataloader(self, train):
    data = self.train if train else self.val
    return torch.utils.data.DataLoader(data, self.batch_size, shuffle=train,
                                       num_workers=self.num_workers)

Para ver cómo funciona esto, carguemos un minibatch de imágenes invocando el método `train_dataloader`. Contiene 64 imágenes.


In [ ]:
X, y = next(iter(data.train_dataloader()))
print(X.shape, X.dtype, y.shape, y.dtype)

Echemos un vistazo al tiempo que lleva leer las imágenes. Aunque es un cargador incorporado, no es muy rápido. Sin embargo, esto es suficiente ya que el procesamiento de imágenes con una red profunda toma bastante más tiempo. Por lo tanto, es lo suficientemente bueno como para que el entrenamiento de una red no esté limitado por E/S.


In [ ]:
tic = time.time()
for X, y in data.train_dataloader():
    continue
f'{time.time() - tic:.2f} sec'

## Visualización
A menudo vamos a utilizar el conjunto de datos de Fashion-MNIST. Una función de conveniencia `show_images` se puede utilizar para visualizar las imágenes y las etiquetas asociadas. Saltar los detalles de la implementación, sólo mostramos la interfaz a continuación: sólo necesitamos saber cómo invocar `d2l.show_images` en lugar de cómo funciona para tales funciones de utilidad.


In [ ]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):  #@save
    """Trace una lista de imágenes."""
    return d2l.show_images(imgs, num_rows, num_cols, titles, scale)

En general, es una buena idea visualizar e inspeccionar los datos en los que estás entrenando. Los humanos son muy buenos en detectar rarezas y por eso, la visualización sirve como una salvaguardia adicional contra errores y errores en el diseño de experimentos. Aquí están ** las imágenes y sus etiquetas correspondientes** (en texto) para los primeros ejemplos en el conjunto de datos de entrenamiento.


In [ ]:
@d2l.add_to_class(FashionMNIST)  #@save
def visualize(self, batch, nrows=1, ncols=8, labels=[]):
    X, y = batch
    if not labels:
        labels = self.text_labels(y)
    d2l.show_images(X.squeeze(1), nrows, ncols, titles=labels)
batch = next(iter(data.val_dataloader()))
data.visualize(batch)

Ahora estamos listos para trabajar con el conjunto de datos de Fashion-MNIST en las secciones que siguen.

## Resumen
Ahora tenemos un conjunto de datos un poco más realista que utilizar para la clasificación. Fashion-MNIST es un conjunto de datos de clasificación de prendas de vestir que consiste en imágenes que representan 10 categorías. Usaremos este conjunto de datos en secciones y capítulos posteriores para evaluar diversos diseños de red, desde un modelo lineal simple hasta redes residuales avanzadas. Como hacemos comúnmente con las imágenes, las leemos como un tensor de forma (tamaño de lote, número de canales, altura, anchura). Por ahora, sólo tenemos un canal, ya que las imágenes son escala de grises (la visualización anterior utiliza una paleta de color falsa para mejorar la visibilidad).

Por último, los iteradores de datos son un componente clave para un rendimiento eficiente. Por ejemplo, podemos utilizar GPUs para una descompresión de imágenes eficiente, transcodificación de vídeo u otro preprocesamiento. Siempre que sea posible, debe confiar en iteradores de datos bien implementados que exploten la computación de alto rendimiento para evitar ralentizar su bucle de entrenamiento.

## Ejercicios
1. ¿La reducción del `batch_size` (por ejemplo, a 1) afecta el rendimiento de lectura?
1. El rendimiento del iterador de datos es importante. ¿Cree que la implementación actual es lo suficientemente rápida? Explore varias opciones para mejorarla. Use un perfilador de sistema para averiguar dónde están los cuellos de botella.
1. Echa un vistazo a la documentación API en línea del framework. ¿Qué otros conjuntos de datos están disponibles?


[Debate del original](https://discuss.d2l.ai/t/49)
